# Reinforcement Component Catalogue - Architecture Design

## Motivation

Current implementation in `cs_design` has simplified reinforcement handling:
- Direct material specification (steel grade, properties)
- Limited to simple symmetric layouts
- No structured product catalog

**Real-world requirements:**
1. **Standardized Products**: Certified reinforcement components with known properties
2. **Multiple Types**: 
   - Steel rebars (various diameters, grades)
   - Carbon bars (COMBAR, etc.)
   - Textile reinforcement (Solidian fabrics, grids)
3. **Material Associations**: Each product links to validated stress-strain behavior
4. **Safety Factors**: Convert characteristic → design values
5. **Structured Catalog**: Easy lookup by shape, material, diameter

## Proposed Architecture

```
bmcs_cross_section/
├── matmod/              # Material models (stress-strain curves)
│   ├── steel_reinforcement.py
│   ├── carbon_reinforcement.py
│   └── textile_reinforcement.py
├── cs_components/       # NEW: Product catalog layer
│   ├── __init__.py
│   ├── reinforcement_catalog.py    # Catalog infrastructure
│   ├── steel_rebars.py             # Standard steel bar products
│   ├── carbon_bars.py              # COMBAR products
│   ├── textile_products.py         # Solidian fabrics
│   └── component_base.py           # Base classes
└── cs_design/           # Cross-section design (uses components)
    ├── shapes.py
    ├── reinforcement.py
    └── cross_section.py
```

## This Notebook

Explores and prototypes:
1. Component base classes
2. Catalog structure (pandas-based)
3. Steel rebar catalog
4. Carbon bar (COMBAR) catalog
5. Textile reinforcement catalog
6. Characteristic → Design value conversion
7. Integration with existing matmod

## 1. Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Tuple, Any
from abc import ABC, abstractmethod

# Existing matmod
from bmcs_cross_section.matmod.steel_reinforcement import SteelReinforcement, create_steel

print("✓ Imports successful")

## 2. Review Current Reinforcement Implementation

Let's check what's in the existing `cs_layout` code that we should preserve.

In [ ]:
# Check if cs_layout exists and what it contains
import os
from pathlib import Path

base_path = Path('/home/rch/Coding/bmcs_cross_section/bmcs_cross_section')

# Check for cs_layout
cs_layout_path = base_path / 'cs_design' / 'cs_layout.py'
if cs_layout_path.exists():
    print("Found cs_layout.py")
    with open(cs_layout_path, 'r') as f:
        content = f.read()
        # Look for key classes/concepts
        if 'rebar' in content.lower():
            print("  - Contains 'rebar' concept")
        if 'layer' in content.lower():
            print("  - Contains 'layer' concept")
        if 'textile' in content.lower():
            print("  - Contains 'textile' concept")
else:
    print("cs_layout.py not found in cs_design")

# Check what's in cs_design
cs_design_path = base_path / 'cs_design'
if cs_design_path.exists():
    print(f"\ncs_design directory contents:")
    for item in sorted(cs_design_path.glob('*.py')):
        print(f"  - {item.name}")

## 3. Component Base Architecture

Define base classes for reinforcement components.

In [ ]:
@dataclass
class ReinforcementComponent(ABC):
    """
    Base class for all reinforcement components.
    
    A component represents a standardized product with:
    - Geometric properties (area, diameter, etc.)
    - Material behavior (stress-strain curve)
    - Safety factors for design
    - Manufacturer/certification info
    """
    
    # Product identification
    product_id: str
    name: str
    manufacturer: str
    
    # Shape category
    shape_type: str = ''  # 'bar', 'mat', 'textile', 'grid'
    
    # Material type
    material_type: str = ''  # 'steel', 'carbon', 'glass', 'basalt', 'aramid'
    
    # Geometric properties
    nominal_diameter: Optional[float] = None  # [mm] for bars
    area: float = 0.0  # [mm²] cross-sectional area
    
    # Material properties (characteristic values)
    f_tk: float = 0.0  # [MPa] characteristic tensile strength
    E: float = 0.0     # [MPa] elastic modulus
    eps_uk: float = 0.0  # [-] characteristic ultimate strain
    
    # Safety factors (EC2 typical values)
    gamma_s: float = 1.15  # Partial safety factor for reinforcement
    
    # Optional: link to full material model (use Any for flexibility)
    matmod: Optional[Any] = None
    
    def __post_init__(self):
        """Calculate derived properties."""
        if self.nominal_diameter and self.area == 0.0:
            # Calculate area from diameter for circular bars
            self.area = np.pi * (self.nominal_diameter / 2) ** 2
    
    @property
    def f_td(self) -> float:
        """Design tensile strength [MPa]."""
        return self.f_tk / self.gamma_s
    
    @property
    def eps_ud(self) -> float:
        """Design ultimate strain [-]."""
        return self.eps_uk  # Usually not reduced by gamma_s
    
    @abstractmethod
    def get_design_stress_strain(self, eps: np.ndarray) -> np.ndarray:
        """
        Get design stress-strain curve.
        
        Args:
            eps: Strain array [-]
            
        Returns:
            Stress array [MPa] at design values
        """
        pass
    
    def to_dict(self) -> Dict:
        """Convert to dictionary for catalog."""
        return {
            'product_id': self.product_id,
            'name': self.name,
            'manufacturer': self.manufacturer,
            'shape_type': self.shape_type,
            'material_type': self.material_type,
            'nominal_diameter': self.nominal_diameter,
            'area': self.area,
            'f_tk': self.f_tk,
            'f_td': self.f_td,
            'E': self.E,
            'eps_uk': self.eps_uk,
            'gamma_s': self.gamma_s,
        }

print("✓ ReinforcementComponent base class defined")

## 4. Steel Rebar Components

Standard steel reinforcing bars according to EC2.

In [ ]:
@dataclass
class SteelRebarComponent(ReinforcementComponent):
    """
    Steel reinforcing bar component.
    
    Standard diameters (EC2): 6, 8, 10, 12, 14, 16, 20, 25, 28, 32, 40 mm
    Grades: B500A, B500B, B500C (f_yk = 500 MPa)
    """
    
    grade: str = 'B500B'  # Steel grade
    
    def __post_init__(self):
        super().__post_init__()
        self.shape_type = 'bar'
        self.material_type = 'steel'
        
        # Create matmod if not provided
        if self.matmod is None:
            self.matmod = create_steel(self.grade)
            self.f_tk = self.matmod.f_sy  # Yield strength as characteristic
            self.E = self.matmod.E_s
            self.eps_uk = self.matmod.eps_ud  # Ultimate strain
    
    def get_design_stress_strain(self, eps: np.ndarray) -> np.ndarray:
        """
        Get design stress-strain curve.
        
        For steel: f_yd = f_yk / gamma_s
        """
        # Get characteristic curve from matmod
        sig_char = self.matmod.get_sig(eps)
        
        # Apply safety factor
        sig_design = sig_char / self.gamma_s
        
        return sig_design


# Create standard steel rebar catalog
def create_steel_rebar_catalog() -> pd.DataFrame:
    """
    Create catalog of standard steel rebars.
    
    Returns:
        DataFrame with all standard products
    """
    catalog = []
    
    # Standard diameters
    diameters = [6, 8, 10, 12, 14, 16, 20, 25, 28, 32, 40]
    
    # Grades
    grades = ['B500A', 'B500B', 'B500C']
    
    for grade in grades:
        for d in diameters:
            product_id = f"REBAR-{grade}-D{d}"
            component = SteelRebarComponent(
                product_id=product_id,
                name=f"Steel Rebar Ø{d} {grade}",
                manufacturer="Standard EC2",
                grade=grade,
                nominal_diameter=d,
                area=np.pi * (d/2)**2,
            )
            catalog.append(component.to_dict())
    
    return pd.DataFrame(catalog)


steel_catalog = create_steel_rebar_catalog()
print(f"✓ Steel rebar catalog created: {len(steel_catalog)} products")
print(f"\nSample entries:")
steel_catalog.head(10)

In [ ]:
# Visualize catalog structure
print("Steel Rebar Catalog by Grade and Diameter:")
print("="*60)
pivot = steel_catalog.pivot_table(
    index='nominal_diameter',
    columns='name',
    values='area',
    aggfunc='first'
)
print(f"\nDiameters available: {steel_catalog['nominal_diameter'].unique()}")
print(f"Grades available: {steel_catalog['name'].str.split().str[-1].unique()}")

## 5. Carbon Bar Components (COMBAR)

Carbon fiber reinforced polymer bars.

In [ ]:
@dataclass
class CarbonBarComponent(ReinforcementComponent):
    """
    Carbon fiber reinforced polymer bar (e.g., COMBAR).
    
    Properties:
    - Linear elastic to failure
    - High tensile strength (1500-2500 MPa)
    - High stiffness (E ≈ 150-170 GPa)
    - No yielding (brittle failure)
    """
    
    product_line: str = 'COMBAR'  # Product line
    
    def __post_init__(self):
        super().__post_init__()
        self.shape_type = 'bar'
        self.material_type = 'carbon'
        # Carbon FRP typically has higher safety factor
        if self.gamma_s == 1.15:  # If default
            self.gamma_s = 1.25  # EC2-like value for FRP
    
    def get_design_stress_strain(self, eps: np.ndarray) -> np.ndarray:
        """
        Linear elastic to failure for carbon bars.
        """
        # Design values
        f_td = self.f_td
        eps_ud = self.eps_ud
        
        # Linear elastic
        sig = np.minimum(self.E * np.abs(eps), f_td)
        sig = np.sign(eps) * sig  # Keep sign
        
        # Limit to ultimate strain
        sig = np.where(np.abs(eps) > eps_ud, 0, sig)
        
        return sig


def create_carbon_bar_catalog() -> pd.DataFrame:
    """
    Create catalog of COMBAR products.
    
    Typical COMBAR diameters: 6, 8, 10, 12, 14, 16, 20, 25, 28, 32 mm
    """
    catalog = []
    
    # COMBAR typical properties
    diameters = [6, 8, 10, 12, 14, 16, 20, 25, 28, 32]
    f_tk_carbon = 2000  # MPa (typical)
    E_carbon = 165000   # MPa
    eps_uk_carbon = f_tk_carbon / E_carbon
    
    for d in diameters:
        product_id = f"COMBAR-D{d}"
        component = CarbonBarComponent(
            product_id=product_id,
            name=f"COMBAR Ø{d} Carbon",
            manufacturer="Schöck",
            product_line='COMBAR',
            nominal_diameter=d,
            area=np.pi * (d/2)**2,
            f_tk=f_tk_carbon,
            E=E_carbon,
            eps_uk=eps_uk_carbon,
            gamma_s=1.25,
        )
        catalog.append(component.to_dict())
    
    return pd.DataFrame(catalog)


carbon_catalog = create_carbon_bar_catalog()
print(f"✓ Carbon bar catalog created: {len(carbon_catalog)} products")
print(f"\nSample entries:")
carbon_catalog.head()

## 6. Textile Reinforcement Components (Solidian)

Grid/fabric reinforcement for textile-reinforced concrete (TRC).

In [ ]:
@dataclass
class TextileReinforcementComponent(ReinforcementComponent):
    """
    Textile reinforcement (grid, fabric).
    
    Properties:
    - Area given per unit width [mm²/m]
    - Can be carbon, glass, basalt, aramid
    - Grid spacing defines effective area
    """
    
    # Textile-specific properties
    area_per_width: float = 0.0  # [mm²/m] cross-sectional area per meter width
    grid_spacing: float = 0.0    # [mm] spacing between rovings
    roving_tex: float = 0.0      # [tex = g/km] roving fineness
    
    def __post_init__(self):
        super().__post_init__()
        self.shape_type = 'textile'
        # Don't call super().__post_init__() diameter calculation
        # Area is specified directly for textiles
    
    def get_design_stress_strain(self, eps: np.ndarray) -> np.ndarray:
        """
        Linear elastic to failure (typical for textile).
        """
        f_td = self.f_td
        eps_ud = self.eps_ud
        
        # Linear elastic
        sig = np.minimum(self.E * np.abs(eps), f_td)
        sig = np.sign(eps) * sig
        
        # Limit to ultimate strain
        sig = np.where(np.abs(eps) > eps_ud, 0, sig)
        
        return sig


def create_textile_catalog() -> pd.DataFrame:
    """
    Create catalog of Solidian textile products.
    
    Common products:
    - Carbon grids (high strength, high stiffness)
    - Glass grids (lower cost)
    - Basalt grids (corrosion resistant)
    """
    catalog = []
    
    # Carbon textile properties
    carbon_products = [
        {'roving': '800tex', 'spacing': 10, 'area_per_m': 107, 'f_tk': 3200, 'E': 240000},
        {'roving': '1600tex', 'spacing': 15, 'area_per_m': 214, 'f_tk': 3200, 'E': 240000},
        {'roving': '3200tex', 'spacing': 20, 'area_per_m': 428, 'f_tk': 3200, 'E': 240000},
    ]
    
    for prod in carbon_products:
        product_id = f"SOLIDIAN-C-{prod['roving']}-{prod['spacing']}"
        eps_uk = prod['f_tk'] / prod['E']
        component = TextileReinforcementComponent(
            product_id=product_id,
            name=f"Solidian GRID Q{prod['spacing']}-C-{prod['roving']}",
            manufacturer="Solidian",
            material_type='carbon',
            area_per_width=prod['area_per_m'],
            area=prod['area_per_m'],  # For compatibility
            grid_spacing=prod['spacing'],
            roving_tex=float(prod['roving'].replace('tex', '')),
            f_tk=prod['f_tk'],
            E=prod['E'],
            eps_uk=eps_uk,
            gamma_s=1.25,
        )
        catalog.append(component.to_dict())
    
    # Glass textile properties
    glass_products = [
        {'roving': '1200tex', 'spacing': 10, 'area_per_m': 168, 'f_tk': 1700, 'E': 72000},
        {'roving': '2400tex', 'spacing': 15, 'area_per_m': 336, 'f_tk': 1700, 'E': 72000},
    ]
    
    for prod in glass_products:
        product_id = f"SOLIDIAN-G-{prod['roving']}-{prod['spacing']}"
        eps_uk = prod['f_tk'] / prod['E']
        component = TextileReinforcementComponent(
            product_id=product_id,
            name=f"Solidian GRID Q{prod['spacing']}-G-{prod['roving']}",
            manufacturer="Solidian",
            material_type='glass',
            area_per_width=prod['area_per_m'],
            area=prod['area_per_m'],
            grid_spacing=prod['spacing'],
            roving_tex=float(prod['roving'].replace('tex', '')),
            f_tk=prod['f_tk'],
            E=prod['E'],
            eps_uk=eps_uk,
            gamma_s=1.30,  # Slightly higher for glass
        )
        catalog.append(component.to_dict())
    
    return pd.DataFrame(catalog)


textile_catalog = create_textile_catalog()
print(f"✓ Textile reinforcement catalog created: {len(textile_catalog)} products")
print(f"\nSample entries:")
textile_catalog

## 7. Combined Catalog

Merge all catalogs into unified database.

In [ ]:
# Combine all catalogs
full_catalog = pd.concat([
    steel_catalog,
    carbon_catalog,
    textile_catalog
], ignore_index=True)

print(f"✓ Full reinforcement catalog: {len(full_catalog)} products")
print(f"\nBreakdown by material type:")
print(full_catalog['material_type'].value_counts())
print(f"\nBreakdown by shape type:")
print(full_catalog['shape_type'].value_counts())

In [ ]:
# Query examples
print("Query Example 1: All Ø16 products")
print("="*60)
d16_products = full_catalog[full_catalog['nominal_diameter'] == 16]
print(d16_products[['product_id', 'name', 'area', 'f_tk', 'f_td']])

print("\nQuery Example 2: All carbon reinforcement")
print("="*60)
carbon_products = full_catalog[full_catalog['material_type'] == 'carbon']
print(carbon_products[['product_id', 'name', 'shape_type', 'f_tk']])

print("\nQuery Example 3: High-strength products (f_tk > 2000 MPa)")
print("="*60)
high_strength = full_catalog[full_catalog['f_tk'] > 2000]
print(high_strength[['product_id', 'name', 'f_tk', 'E']])

## 8. Characteristic vs Design Values

Demonstrate characteristic → design value conversion.

In [ ]:
# Compare characteristic vs design curves for different materials
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Strain range
eps = np.linspace(0, 0.025, 100)

# 1. Steel rebar B500B Ø16
steel_component = SteelRebarComponent(
    product_id='TEST-STEEL',
    name='Steel B500B Ø16',
    manufacturer='Test',
    grade='B500B',
    nominal_diameter=16,
)

sig_char_steel = steel_component.matmod.get_sig(eps)
sig_design_steel = steel_component.get_design_stress_strain(eps)

axes[0].plot(eps * 1000, sig_char_steel, 'b-', linewidth=2, label='Characteristic')
axes[0].plot(eps * 1000, sig_design_steel, 'r--', linewidth=2, label='Design (γ_s=1.15)')
axes[0].set_xlabel('Strain [‰]')
axes[0].set_ylabel('Stress [MPa]')
axes[0].set_title('Steel B500B Ø16')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# 2. Carbon bar COMBAR Ø16
carbon_component = CarbonBarComponent(
    product_id='TEST-CARBON',
    name='COMBAR Ø16',
    manufacturer='Schöck',
    product_line='COMBAR',
    nominal_diameter=16,
    f_tk=2000,
    E=165000,
    eps_uk=2000/165000,
    gamma_s=1.25,
)

sig_char_carbon = carbon_component.E * eps
sig_char_carbon = np.minimum(sig_char_carbon, carbon_component.f_tk)
sig_design_carbon = carbon_component.get_design_stress_strain(eps)

axes[1].plot(eps * 1000, sig_char_carbon, 'b-', linewidth=2, label='Characteristic')
axes[1].plot(eps * 1000, sig_design_carbon, 'r--', linewidth=2, label='Design (γ_s=1.25)')
axes[1].set_xlabel('Strain [‰]')
axes[1].set_ylabel('Stress [MPa]')
axes[1].set_title('COMBAR Carbon Ø16')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

# 3. Comparison at same scale
axes[2].plot(eps * 1000, sig_design_steel, 'b-', linewidth=2, label='Steel B500B (design)')
axes[2].plot(eps * 1000, sig_design_carbon, 'r-', linewidth=2, label='COMBAR (design)')
axes[2].set_xlabel('Strain [‰]')
axes[2].set_ylabel('Stress [MPa]')
axes[2].set_title('Design Curves Comparison')
axes[2].grid(True, alpha=0.3)
axes[2].legend()
axes[2].set_ylim(0, 600)

plt.tight_layout()
plt.show()

print(f"\nSteel: f_yk = {steel_component.f_tk:.0f} MPa → f_yd = {steel_component.f_td:.0f} MPa")
print(f"Carbon: f_tk = {carbon_component.f_tk:.0f} MPa → f_td = {carbon_component.f_td:.0f} MPa")

## 9. Integration with cs_design

Show how components integrate with existing reinforcement layout.

In [ ]:
# Example: Create reinforcement layer using catalog component
from bmcs_cross_section.cs_design import ReinforcementLayer, ReinforcementLayout

# Select component from catalog
component = SteelRebarComponent(
    product_id='REBAR-B500B-D20',
    name='Steel Rebar Ø20 B500B',
    manufacturer='Standard EC2',
    grade='B500B',
    nominal_diameter=20,
)

# Create layer using component properties
# Number of bars: 4 bars
n_bars = 4
total_area = component.area * n_bars

layer = ReinforcementLayer(
    z=50,  # [mm] from bottom
    A_s=total_area,  # Total area of 4Ø20
    material=component.matmod,  # Use component's material model
)

print(f"✓ Created reinforcement layer:")
print(f"  Component: {component.name}")
print(f"  Number of bars: {n_bars}")
print(f"  Single bar area: {component.area:.1f} mm²")
print(f"  Total layer area: {total_area:.1f} mm²")
print(f"  Position: z = {layer.z} mm")
print(f"  Material: {type(layer.material).__name__}")

## 10. Proposed Implementation Structure

Summary of how to implement this in the codebase.

In [ ]:
implementation_plan = """
IMPLEMENTATION PLAN: cs_components Subpackage
============================================

1. CREATE NEW SUBPACKAGE: bmcs_cross_section/cs_components/

   cs_components/
   ├── __init__.py              # Export public API
   ├── component_base.py        # ReinforcementComponent ABC
   ├── reinforcement_catalog.py # Catalog infrastructure
   ├── steel_rebars.py          # SteelRebarComponent + catalog
   ├── carbon_bars.py           # CarbonBarComponent + catalog
   ├── textile_products.py      # TextileReinforcementComponent + catalog
   └── catalog_data/            # Optional: JSON/CSV data files

2. KEY CLASSES:

   ReinforcementComponent (ABC):
   - Base class for all reinforcement products
   - Properties: product_id, name, manufacturer, shape_type, material_type
   - Geometric: nominal_diameter, area
   - Material: f_tk, E, eps_uk (characteristic values)
   - Safety: gamma_s (partial factor)
   - Methods: get_design_stress_strain(), to_dict()
   - Derived: f_td (design strength), eps_ud (design strain)

   SteelRebarComponent:
   - Standard steel rebars per EC2
   - Links to SteelReinforcement matmod
   - Handles bilinear stress-strain with hardening

   CarbonBarComponent:
   - Carbon FRP bars (COMBAR, etc.)
   - Linear elastic to failure
   - Higher safety factor (γ_s = 1.25)

   TextileReinforcementComponent:
   - Grid/fabric reinforcement
   - Area per width [mm²/m]
   - Grid spacing, roving tex

3. CATALOG FUNCTIONS:

   create_steel_rebar_catalog() -> pd.DataFrame
   create_carbon_bar_catalog() -> pd.DataFrame
   create_textile_catalog() -> pd.DataFrame
   get_full_catalog() -> pd.DataFrame

4. QUERY INTERFACE:

   find_component(product_id: str) -> ReinforcementComponent
   search_by_diameter(diameter: float) -> List[ReinforcementComponent]
   search_by_material(material_type: str) -> List[ReinforcementComponent]
   search_by_shape(shape_type: str) -> List[ReinforcementComponent]

5. INTEGRATION WITH cs_design:

   ReinforcementLayer can be created from component:
   
   component = find_component('REBAR-B500B-D20')
   layer = ReinforcementLayer(
       z=50,
       A_s=component.area * n_bars,
       material=component.matmod,
   )

6. STREAMLIT APP ENHANCEMENT:

   - Dropdown to select from catalog instead of manual input
   - Show product properties automatically
   - Display characteristic vs design values
   - Filter by material type, shape, diameter

7. ADVANTAGES:

   ✓ Single source of truth for product properties
   ✓ Easy to extend with new products
   ✓ Automatic design value calculation
   ✓ Traceability (manufacturer, certification)
   ✓ Consistent safety factor handling
   ✓ Integration with existing matmod
   ✓ Pandas-based queries (familiar, powerful)

8. FUTURE EXTENSIONS:

   - Add concrete product catalog (ready-mix grades)
   - Include cost information
   - Add certification/approval data
   - Support custom user products
   - Export/import catalog from JSON/CSV
   - Integration with material testing databases
"""

print(implementation_plan)

## Summary

This notebook has explored the design of a reinforcement component catalog system:

✅ **Base Architecture**: `ReinforcementComponent` abstract class with common properties  
✅ **Steel Rebars**: Standard EC2 rebars with grades and diameters  
✅ **Carbon Bars**: COMBAR-like products with linear elastic behavior  
✅ **Textile Reinforcement**: Solidian grids with area-per-width specification  
✅ **Unified Catalog**: Pandas DataFrame for easy querying  
✅ **Design Values**: Automatic characteristic → design conversion with safety factors  
✅ **Integration**: Compatible with existing cs_design components  

**Next Steps:**
1. Implement `cs_components` subpackage in codebase
2. Enhance Streamlit app with catalog selection
3. Add concrete product catalog
4. Update documentation with new workflow

The catalog approach provides:
- **Traceability**: Know exactly which product is being used
- **Consistency**: Standardized properties across projects
- **Safety**: Built-in design value calculation
- **Extensibility**: Easy to add new products and materials